# Notebook 3: Model Development - Feature Engineering & Embeddings

This notebook focuses on preparing the combined (malicious and benign)
transaction data for deep learning model training.

**Key Stages:**
1.  **Data Loading**: Load preprocessed malicious data and raw benign data.
2.  **Benign Data Preprocessing**: Apply consistent preprocessing to benign data.
3.  **Labeling & Merging**: Label datasets and combine them.
4.  **Feature Engineering**: Prepare features like function signatures for
    embedding on the combined dataset.
5.  **Final Type Coercion and Saving of `balanced_dataset.parquet`**:
    This dataset includes `function_sig_idx` and all other final features.
6.  **Data Splitting**: Divide the final `combined_df` into training,
    validation, and test sets.
7.  **Feature Scaling**: Scale numerical features.
8.  **Saving Data Splits**: Save the prepared X and y splits for the
    next notebook (`04_...`).

## 1. Imports and Path Configuration

In [ ]:
# ----- CODE CELL: Imports and Path Configuration -----
# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# ML specific imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # For feature scaling
from eth_utils import decode_hex # For function signature processing

# --- Path Configuration ---
# Assuming this notebook/script is in 'EthMalViz/notebooks/'
PROJECT_ROOT = Path('..')
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORTS_FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

# Ensure output directories exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_SPLITS_DIR_V1 = PROCESSED_DATA_DIR / "data_splits_v1" # For saving splits
DATA_SPLITS_DIR_V1.mkdir(parents=True, exist_ok=True)


print("Paths configured.")
print(f"  Processed data directory: {PROCESSED_DATA_DIR}")
print(f"  Raw data directory: {RAW_DATA_DIR}")
print(f"  Models directory: {MODEL_DIR}")
print(f"  Figures directory: {REPORTS_FIGURES_DIR}")
print(f"  Data splits V1 directory: {DATA_SPLITS_DIR_V1}")

## 2. Data Loading

Load the preprocessed malicious transactions (from
`01_data_preprocessing.ipynb`) and the raw benign transactions (collected by
`02_benign_data_collection.ipynb`).

In [ ]:
# ----- CODE CELL: Data Loading -----
# --- Load Malignant (Preprocessed) Dataset ---
malignant_parquet_file = 'transactions_cleaned.parquet'
malignant_parquet_path = PROCESSED_DATA_DIR / malignant_parquet_file
print(f"Attempting to load malignant data from: {malignant_parquet_path}")
try:
    tx_df_malicious = pd.read_parquet(malignant_parquet_path)
    print(
        "Successfully loaded preprocessed malicious data. "
        f"Shape: {tx_df_malicious.shape}"
    )
except FileNotFoundError:
    print(f"ERROR: Malignant data file not found: {malignant_parquet_path}")
    print("Please ensure '01_data_preprocessing.ipynb' was run successfully.")
    raise
except Exception as e:
    print(f"Error loading malignant data: {e}")
    raise

# --- Load Raw Benign Dataset ---
benign_raw_dir = RAW_DATA_DIR / 'benign_transactions'
benign_json_files = list(benign_raw_dir.glob("*.json"))
print(
    f"\nFound {len(benign_json_files)} benign transaction JSON files "
    f"in {benign_raw_dir}"
)

benign_df_raw = pd.DataFrame()
if benign_json_files:
    all_benign_tx_list = []
    for file_path in benign_json_files:
        with open(file_path, 'r') as f:
            try:
                data = json.load(f)
                if isinstance(data, list):
                    all_benign_tx_list.extend(data)
            except json.JSONDecodeError:
                print(f"Warning: Could not decode JSON from {file_path.name}")
    
    if all_benign_tx_list:
        benign_df_raw = pd.DataFrame(all_benign_tx_list)
        print(
            f"Successfully loaded {len(benign_df_raw)} raw "
            "benign transactions."
        )
    elif not all_benign_tx_list and benign_json_files:
        print("Warning: Benign JSON files found but no data was loaded.")
else:
    print(
        f"WARNING: No benign transaction JSON files found in "
        f"{benign_raw_dir}. Dataset will be imbalanced if proceeding."
    )

## 3. Benign Data Preprocessing

Apply the same preprocessing steps to the raw benign transactions as were
applied to the malicious transactions in `01_data_preprocessing.ipynb`.
This ensures that both datasets have a consistent structure and features
before they are combined.

In [ ]:
# ----- CODE CELL: Benign Data Preprocessing -----
print(
    f"\nStarting preprocessing for benign_df_raw "
    f"(Shape: {benign_df_raw.shape})"
)
benign_df_processed = pd.DataFrame()

if not benign_df_raw.empty:
    benign_df_processed = benign_df_raw.copy()

    def safe_hex_to_int_or_float(value_to_convert): # Renamed from file
        if isinstance(value_to_convert, (int, float)):
            return value_to_convert
        if isinstance(value_to_convert, str):
            if value_to_convert.startswith('0x'):
                try:
                    return int(value_to_convert, 16)
                except ValueError:
                    return np.nan
            else: 
                try:
                    return int(value_to_convert)
                except ValueError:
                    try:
                        return float(value_to_convert)
                    except ValueError:
                        return np.nan
        return np.nan

    numeric_cols_in_raw_json = [
        'blockNumber', 'timeStamp', 'nonce', 'transactionIndex', 'value',
        'gas', 'gasPrice', 'gasUsed', 'confirmations'
    ]
    for col in numeric_cols_in_raw_json:
        if col in benign_df_processed.columns:
            benign_df_processed[col] = benign_df_processed[col].apply(
                safe_hex_to_int_or_float
            )
            benign_df_processed[col] = pd.to_numeric(
                benign_df_processed[col], errors='coerce'
            )

    if 'value' in benign_df_processed.columns:
        benign_df_processed['value'] = benign_df_processed[
            'value'
        ].fillna(0)
    if 'input' in benign_df_processed.columns:
        benign_df_processed['input'] = benign_df_processed[
            'input'
        ].fillna('0x')
    else:
        benign_df_processed['input'] = '0x'

    if 'timeStamp' in benign_df_processed.columns:
        benign_df_processed['timeStamp_numeric'] = pd.to_numeric(
            benign_df_processed['timeStamp'], errors='coerce'
        ).fillna(0).astype(np.int64)
        benign_df_processed['timestamp'] = pd.to_datetime(
            benign_df_processed['timeStamp_numeric'], 
            unit='s', 
            errors='coerce'
        )
        benign_df_processed['hour_of_day'] = benign_df_processed[
            'timestamp'
        ].dt.hour
    else:
        benign_df_processed['timestamp'] = pd.NaT
        benign_df_processed['hour_of_day'] = np.nan
        
    if ('from' in benign_df_processed.columns and 
            'to' in benign_df_processed.columns):
        benign_df_processed['from_to_pair'] = (
            benign_df_processed['from'].astype(str) + '_' + 
            benign_df_processed['to'].astype(str)
        )
    else:
        benign_df_processed['from_to_pair'] = np.nan

    if ('gasUsed' in benign_df_processed.columns and 
            'gas' in benign_df_processed.columns):
        benign_df_processed['gasUsed_numeric'] = pd.to_numeric(
            benign_df_processed['gasUsed'], errors='coerce'
        ).fillna(0)
        benign_df_processed['gas_numeric'] = pd.to_numeric(
            benign_df_processed['gas'], errors='coerce'
        ).fillna(0)
        benign_df_processed['gas_efficiency'] = benign_df_processed.apply(
            lambda row: (row['gasUsed_numeric'] / row['gas_numeric'] 
                         if row['gas_numeric'] != 0 else 0), axis=1
        )
        benign_df_processed['gas_efficiency'] = benign_df_processed[
            'gas_efficiency'
        ].replace([np.inf, -np.inf], 0)
    else:
        benign_df_processed['gas_efficiency'] = np.nan

    def extract_sig_from_input_nb03(input_data: str): # Renamed
        if not isinstance(input_data, str) or \
           not input_data.startswith('0x'):
            return None
        if len(input_data) >= 10:
            hex_sig_part = input_data[2:10]
            try:
                return decode_hex(hex_sig_part)
            except Exception:
                return None
        return None

    if 'input' in benign_df_processed.columns:
        benign_df_processed['function_sig_bytes'] = benign_df_processed[
            'input'
        ].apply(extract_sig_from_input_nb03)
        benign_df_processed['function_sig_hex'] = benign_df_processed[
            'function_sig_bytes'
        ].apply(lambda b: '0x' + b.hex() if isinstance(b, bytes) else None)
    else:
        benign_df_processed['function_sig_bytes'] = None
        benign_df_processed['function_sig_hex'] = None
        
    benign_df_processed['is_malicious'] = 0
    print("Benign data preprocessing completed.")
else:
    print("benign_df_raw is empty. No benign data to preprocess.")

## 4. Labeling of Malicious Source Data (`tx_df_malicious`)

Ensure `tx_df_malicious` has the 'is_malicious' label correctly applied,
using `addresses_darklist.json`.

In [ ]:
# ----- CODE CELL: Labeling tx_df_malicious -----
blocklist_file = (
    PROJECT_ROOT / 'data' / 'external' / 
    'blocklists' / 'addresses_darklist.json'
)
print(f"Loading malicious addresses from: {blocklist_file}")
malicious_addrs_set = set() # Renamed variable

try:
    with open(blocklist_file, 'r') as f_blocklist:
        raw_blocklist_content = json.load(f_blocklist)
    for item_addr in raw_blocklist_content: # Renamed loop variable
        if isinstance(item_addr, dict) and 'address' in item_addr:
            malicious_addrs_set.add(item_addr['address'].lower())
        elif isinstance(item_addr, str):
            malicious_addrs_set.add(item_addr.lower())
    print(
        f"Loaded {len(malicious_addrs_set)} unique malicious addresses."
    )
except Exception as e_blocklist:
    print(f"Error loading malicious address blocklist: {e_blocklist}")

if malicious_addrs_set:
    tx_df_malicious['is_malicious'] = tx_df_malicious.apply(
        lambda r: 1 if ( # Renamed row to r
            str(r.get('from', '')).lower() in malicious_addrs_set or 
            str(r.get('to', '')).lower() in malicious_addrs_set
        ) else 0, 
        axis=1
    )
elif 'is_malicious' not in tx_df_malicious.columns:
    print(
        "WARNING: Blocklist empty and 'is_malicious' not in "
        "tx_df_malicious. Labeling all as 1 (assuming source is malicious)."
    )
    tx_df_malicious['is_malicious'] = 1

print("\n'is_malicious' column ensured/created in tx_df_malicious.")
if 'is_malicious' in tx_df_malicious.columns:
    print("Distribution in tx_df_malicious BEFORE merging:")
    print(
        tx_df_malicious['is_malicious'].value_counts(
            normalize=True
        ).mul(100).round(2).astype(str) + '%'
    )

## 5. Dataset Merging and Column Alignment

Combine the preprocessed malicious (`tx_df_malicious`) and benign
(`benign_df_processed`) datasets. `tx_df_malicious` serves as the
reference for column structure.

In [ ]:
# ----- CODE CELL: Dataset Merging and Column Alignment -----
print(
    f"\nShapes before concat -> tx_df_malicious: {tx_df_malicious.shape}", 
    end=""
)
if not benign_df_processed.empty:
    print(
        f", benign_df_processed: {benign_df_processed.shape}"
    )
    print("Aligning columns of benign_df_processed...")
    for col_align in tx_df_malicious.columns: # Renamed loop variable
        if col_align not in benign_df_processed.columns:
            if pd.api.types.is_numeric_dtype(tx_df_malicious[col_align]):
                benign_df_processed[col_align] = np.nan
            elif pd.api.types.is_datetime64_any_dtype(tx_df_malicious[col_align]):
                benign_df_processed[col_align] = pd.NaT
            else: # object/string
                benign_df_processed[col_align] = None
    
    benign_df_aligned = benign_df_processed.reindex(
        columns=tx_df_malicious.columns
    )
    combined_df = pd.concat(
        [tx_df_malicious, benign_df_aligned], ignore_index=True
    )
else:
    print("\nbenign_df_processed is empty. Using only tx_df_malicious.")
    combined_df = tx_df_malicious.copy()
print(f"Shape after concat: {combined_df.shape}")

## 6. Feature Engineering on Combined Dataset (Function Signature Index)

Create integer indices for `function_sig_hex` from the entire
`combined_df` to build a global vocabulary for the embedding layer.

In [ ]:
# ----- CODE CELL: Feature Engineering - Function Signature Index -----
if 'function_sig_hex' not in combined_df.columns:
    print(
        "ERROR: 'function_sig_hex' column missing from combined_df! "
        "This column is crucial for model input. "
        "Check preprocessing steps of both malicious and benign dataframes."
    )
    # Add a placeholder if critically needed, but this is an error.
    # combined_df['function_sig_hex'] = None 
    # combined_df['function_sig_idx'] = 0 # Placeholder index
    # sig_vocab_size_final = 1 # Placeholder vocab size
    # UNKNOWN_TOKEN_COMBINED = '<UNKNOWN_SIGNATURE>' # Ensure defined
    # sig_to_idx_final_map = {UNKNOWN_TOKEN_COMBINED: 0} # Placeholder map
    # idx_to_sig_final_map = {0: UNKNOWN_TOKEN_COMBINED} # Placeholder map

else:
    print("\nProcessing 'function_sig_hex' for embedding in combined_df...")
    UNKNOWN_TOKEN_COMBINED = '<UNKNOWN_SIGNATURE>'
    
    combined_df['function_sig_processed'] = combined_df[
        'function_sig_hex'
    ].fillna(UNKNOWN_TOKEN_COMBINED)
    
    unique_sigs_list_final = combined_df[ # Renamed variable
        'function_sig_processed'
    ].unique().tolist()
    
    sig_to_idx_final_map = { # Use a distinct name
        sig_item: idx for idx, sig_item in enumerate(unique_sigs_list_final)
    }
    idx_to_sig_final_map = { # Use a distinct name
        idx: sig_item for sig_item, idx in sig_to_idx_final_map.items()
    }
    sig_vocab_size_final = len(unique_sigs_list_final)
    
    print(
        f"Final vocabulary size for function signatures: "
        f"{sig_vocab_size_final}"
    )
    combined_df['function_sig_idx'] = combined_df[ # THIS CREATES THE COLUMN
        'function_sig_processed'
    ].map(sig_to_idx_final_map)

    # Save the final, comprehensive mapping
    path_final_sig_map = MODEL_DIR / 'function_signature_map_final.json'
    data_final_map = {
        'sig_to_idx': sig_to_idx_final_map,
        'idx_to_sig': idx_to_sig_final_map,
        'vocab_size': sig_vocab_size_final,
        'unknown_token': UNKNOWN_TOKEN_COMBINED
    }
    try:
        with open(path_final_sig_map, 'w') as f_map_out: # Renamed variable
            json.dump(data_final_map, f_map_out, indent=4)
        print(f"Saved final signature mapping to {path_final_sig_map}")
    except Exception as e_map_save: # Renamed variable
        print(f"Error saving final signature mapping: {e_map_save}")

## 7. Final Type Coercion and `balanced_dataset.parquet` Saving

Ensure all columns in `combined_df` have data types compatible with
Parquet and suitable for model input. The `function_sig_idx` column
created above is now included.

In [ ]:
# ----- CODE CELL: Final Type Coercion and Saving -----
# List of columns that should ideally be integers
list_final_int_cols = [ # Renamed variable
    'blockNumber', 'timeStamp', 'nonce', 'transactionIndex', 'value', 
    'gas', 'gasPrice', 'gasUsed', 'confirmations', 'is_malicious', 
    'hour_of_day', 'function_sig_idx' # CRITICAL: function_sig_idx added
]
print("\nPerforming final type coercions on combined_df...")
for col_coerce in list_final_int_cols: # Renamed loop variable
    if col_coerce in combined_df.columns:
        combined_df[col_coerce] = pd.to_numeric(
            combined_df[col_coerce], errors='coerce'
        )
        if combined_df[col_coerce].isna().sum() > 0:
            fill_val = 0 # Default fill
            if col_coerce == 'function_sig_idx' and 'sig_to_idx_final_map' in locals():
                # Use index of UNKNOWN_TOKEN if map exists
                fill_val = sig_to_idx_final_map.get(UNKNOWN_TOKEN_COMBINED, 0)
            combined_df[col_coerce] = combined_df[col_coerce].fillna(fill_val)
        try:
            combined_df[col_coerce] = combined_df[col_coerce].astype(np.int64)
        except ValueError: 
            try: 
                combined_df[col_coerce] = combined_df[col_coerce].astype(
                    np.float64
                ).astype(np.int64) # Try via float
            except Exception as e_final_coerce:
                print(
                    f"  WARNING: Could not convert '{col_coerce}' "
                    f"to int64: {e_final_coerce}."
                )
# Ensure 'timestamp' (datetime) is consistent
if 'timestamp' in combined_df.columns:
    combined_df['timestamp'] = pd.to_datetime(
        combined_df['timestamp'], errors='coerce'
    )
# Float columns
if 'gas_efficiency' in combined_df.columns:
    combined_df['gas_efficiency'] = pd.to_numeric(
        combined_df['gas_efficiency'], errors='coerce'
    ).fillna(0.0)
# Ensure 'function_sig_bytes' contains only bytes or None
if 'function_sig_bytes' in combined_df.columns:
    combined_df['function_sig_bytes'] = combined_df[
        'function_sig_bytes'
    ].apply(lambda x_bytes: x_bytes if isinstance(x_bytes, bytes) else None) # Renamed var

# Drop intermediate processing columns before saving
cols_to_drop = [
    'timeStamp_numeric', 'gasUsed_numeric', 
    'gas_numeric', 'function_sig_processed' # From benign processing
]
# Also drop if they exist from malicious_df processing (though they might have different names)
# For safety, check if they exist before dropping
existing_cols_to_drop = [c for c in cols_to_drop if c in combined_df.columns]
if existing_cols_to_drop:
    combined_df = combined_df.drop(columns=existing_cols_to_drop)
    print(f"\nDropped intermediate columns: {existing_cols_to_drop}")


print("\nCombined_df dtypes sample before saving final parquet:")
# Display a sample of dtypes to verify
print(combined_df.dtypes.sample(min(15, len(combined_df.columns))).sort_index().to_string())


# Shuffle before final save (optional, but good practice if not split yet)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(
    drop=True
)
path_output_balanced = PROCESSED_DATA_DIR / "balanced_dataset.parquet"
try:
    print(
        f"\nSaving final combined_df (Shape: {combined_df.shape}) "
        f"to {path_output_balanced}"
    )
    combined_df.to_parquet(
        path_output_balanced, index=False, engine='pyarrow'
    )
    print(f"Successfully saved final dataset to {path_output_balanced}")
except Exception as e_save_parquet:
    print(f"ERROR saving final combined_df to Parquet: {e_save_parquet}")
    print("\nCombined_df.info() for debugging:")
    combined_df.info()

## 8. Final Class Distribution Analysis and Data Splitting

Analyze the class distribution of the `combined_df` (loaded from the
just-saved `balanced_dataset.parquet` or using the in-memory `combined_df`).
Then, split it into training, validation, and test sets. Numerical
features will be scaled.

In [ ]:
# ----- CODE CELL: Final Class Distribution, Splitting, and Scaling -----
# --- 8.1. Final Class Distribution ---
# This section assumes 'combined_df' is the finalized DataFrame
if 'is_malicious' in combined_df.columns:
    print("\nTarget variable 'is_malicious' in final combined_df.")
    raw_counts_final = combined_df['is_malicious'].value_counts() # Renamed
    print("\nRaw class counts in final combined_df:")
    print(raw_counts_final.to_markdown())

    norm_dist_final = combined_df[ # Renamed
        'is_malicious'
    ].value_counts(normalize=True) * 100
    print("\nNormalized class distribution in final combined_df:")
    print(f"  Benign (0): {norm_dist_final.get(0, 0):.2f}%")
    print(f"  Malicious (1): {norm_dist_final.get(1, 0):.2f}%")
    
    total_txns_final = len(combined_df) # Renamed
    total_mal_final = combined_df['is_malicious'].sum() # Renamed
    print(f"\nTotal transactions in final combined_df: {total_txns_final:,}")
    print(
        "Total malicious transactions in final combined_df: "
        f"{int(total_mal_final):,}"
    )

    # Visualization
    if not combined_df.empty:
        plt.figure(figsize=(8, 6))
        ax_final = sns.countplot( # Renamed
            x='is_malicious', 
            data=combined_df, 
            hue='is_malicious',
            palette=['skyblue', 'salmon'], 
            legend=False 
        )
        plt.title(
            'Class Distribution in Final Combined Dataset for Modeling', 
            fontsize=15
        )
        plt.xlabel(
            'Transaction Class (0: Benign, 1: Malicious)', fontsize=12
        )
        plt.ylabel('Number of Transactions', fontsize=12)
        plt.xticks(ticks=[0, 1], labels=['Benign (0)', 'Malicious (1)'])
        
        for p_item in ax_final.patches: # Renamed loop variable
            h = p_item.get_height() # Renamed
            ax_final.text(
                p_item.get_x() + p_item.get_width() / 2.,
                h + (total_txns_final * 0.005), 
                f'{int(h)}\n({h/total_txns_final:.1%})',
                ha="center", va="bottom", fontsize=10
            )
        plt.tight_layout()
        path_fig_final = REPORTS_FIGURES_DIR / 'final_combined_class_dist.png'
        try:
            plt.savefig(path_fig_final, dpi=300)
            print(f"\nSaved final class distribution plot to {path_fig_final}")
        except Exception as e_fig_save: # Renamed
            print(f"Error saving final plot: {e_fig_save}")
        plt.show()
else: 
    print(
        "ERROR: 'is_malicious' column missing in combined_df for "
        "final analysis."
    )

# --- 8.2. Define Features for Model ---
# These are the columns from the *final* combined_df
list_numerical_features_model = [ # Renamed
    'value', 'gas', 'gasPrice', 'gasUsed', 'confirmations', 
    'hour_of_day', 'gas_efficiency', 'blockNumber', 'nonce', 
    'transactionIndex'
]
list_numerical_features_model = [ # Ensure they exist
    c for c in list_numerical_features_model if c in combined_df.columns
]
# THIS IS THE KEY COLUMN that was missing
name_categorical_idx_feature_model = 'function_sig_idx' # Renamed
name_target_variable_model = 'is_malicious' # Renamed

# Critical check: ensure function_sig_idx exists *before* selecting X and y
if name_categorical_idx_feature_model not in combined_df.columns:
    raise KeyError(
        f"FATAL ERROR: Column '{name_categorical_idx_feature_model}' "
        "is NOT in the final combined_df. Check Section 6."
    )

# Handle NaNs in selected features (should be minimal after prior steps)
for c_fill in list_numerical_features_model: # Renamed
    if combined_df[c_fill].isna().sum() > 0: 
        combined_df[c_fill] = combined_df[c_fill].fillna(0)
if combined_df[name_categorical_idx_feature_model].isna().sum() > 0:
    # This should ideally not happen if UNKNOWN_TOKEN was handled
    # when creating function_sig_idx. Default to index 0 if it does.
    idx_unknown_fill = 0
    if 'sig_to_idx_final_map' in locals(): # Check if map exists
        idx_unknown_fill = sig_to_idx_final_map.get(UNKNOWN_TOKEN_COMBINED,0)
    combined_df[name_categorical_idx_feature_model] = combined_df[
        name_categorical_idx_feature_model
    ].fillna(idx_unknown_fill)

if combined_df[name_target_variable_model].isna().sum() > 0: 
    combined_df[name_target_variable_model] = combined_df[
        name_target_variable_model
    ].fillna(0) # Should not happen for target

X_numerical_model = combined_df[list_numerical_features_model] # Renamed
X_categorical_idx_model = combined_df[[name_categorical_idx_feature_model]] # Renamed
y_model = combined_df[name_target_variable_model] # Renamed

# --- 8.3. Split Data ---
X_num_train, X_num_temp, \
X_cat_idx_train, X_cat_idx_temp, \
y_train, y_temp = train_test_split(
    X_numerical_model, 
    X_categorical_idx_model, 
    y_model, 
    test_size=0.3, 
    random_state=42, 
    stratify=y_model
)
X_num_val, X_num_test, \
X_cat_idx_val, X_cat_idx_test, \
y_val, y_test = train_test_split(
    X_num_temp, 
    X_cat_idx_temp, 
    y_temp, 
    test_size=0.5, 
    random_state=42, 
    stratify=y_temp
)
print("\nData splitting for modeling completed.")

# --- 8.4. Scale Numerical Features ---
scaler_final = StandardScaler() # Renamed
X_num_train_scaled = scaler_final.fit_transform(X_num_train)
X_num_val_scaled = scaler_final.transform(X_num_val)
X_num_test_scaled = scaler_final.transform(X_num_test)
print("\nNumerical features for modeling scaled.")

# --- 8.5. Save Data Splits for Notebook 04 ---
# Convert scaled arrays to DataFrames to preserve column names for Parquet
df_X_num_train_scaled = pd.DataFrame( # Renamed
    X_num_train_scaled, columns=list_numerical_features_model
)
df_X_num_val_scaled = pd.DataFrame(
    X_num_val_scaled, columns=list_numerical_features_model
)
df_X_num_test_scaled = pd.DataFrame(
    X_num_test_scaled, columns=list_numerical_features_model
)

# y_train, y_val, y_test are Series, convert to DataFrame for Parquet
df_y_train = y_train.to_frame(name=name_target_variable_model) # Renamed
df_y_val = y_val.to_frame(name=name_target_variable_model)
df_y_test = y_test.to_frame(name=name_target_variable_model)

# Save each split component
df_X_num_train_scaled.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_num_train_scaled_v1.parquet", index=False
)
df_X_num_val_scaled.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_num_val_scaled_v1.parquet", index=False
)
df_X_num_test_scaled.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_num_test_scaled_v1.parquet", index=False
)

# X_cat_idx_train, _val, _test are already DataFrames
X_cat_idx_train.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_cat_idx_train_v1.parquet", index=False
)
X_cat_idx_val.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_cat_idx_val_v1.parquet", index=False
)
X_cat_idx_test.to_parquet(
    DATA_SPLITS_DIR_V1 / "X_cat_idx_test_v1.parquet", index=False
)

df_y_train.to_parquet(DATA_SPLITS_DIR_V1 / "y_train_v1.parquet", index=False)
df_y_val.to_parquet(DATA_SPLITS_DIR_V1 / "y_val_v1.parquet", index=False)
df_y_test.to_parquet(DATA_SPLITS_DIR_V1 / "y_test_v1.parquet", index=False)

print(
    f"\nAll V1 data splits saved to Parquet files in {DATA_SPLITS_DIR_V1}"
)

--- End of Notebook 3: Data Preparation for Model V1 ---

The next notebook (`04_model_definition_and_training.ipynb`) will load
these saved data splits (`X_num_..._scaled_v1.parquet`, 
`X_cat_idx_..._v1.parquet`, `y_..._v1.parquet`) and focus on defining,
training, and evaluating the V1 model.